# Masterclass: Production AI Agents with LangChain & Gemini 2.0 Flash

This notebook demonstrates production-grade AI Agent patterns in LangChain:
1. Custom tools with Pydantic input validation schemas.
2. Integrating pre-built community tools (Wikipedia).
3. Tool-calling agent creation with ChatGoogleGenerativeAI.
4. Conversational memory management with chat_history.
5. Intermediate steps inspection & tool error recovery.

### Pattern 1: Tools with Pydantic Input Schemas

Using Pydantic BaseModel for args_schema enforces strict argument types, field descriptions, and default values for the LLM tool-calling engine.

In [1]:
import os
from typing import Optional
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

load_dotenv(find_dotenv())

# Define Pydantic Schema for Calculator Input
class MathOperationInput(BaseModel):
    num1: float = Field(description="First number for the mathematical calculation")
    num2: float = Field(description="Second number for the mathematical calculation")
    operation: str = Field(description="Operation to perform: 'add', 'subtract', 'multiply', or 'divide'")

@tool(args_schema=MathOperationInput)
def calculate(num1: float, num2: float, operation: str) -> str:
    """Perform precise arithmetic operations (add, subtract, multiply, divide) on two numbers."""
    op = operation.lower().strip()
    if op == "add":
        return f"Result: {num1 + num2}"
    elif op == "subtract":
        return f"Result: {num1 - num2}"
    elif op == "multiply":
        return f"Result: {num1 * num2}"
    elif op == "divide":
        if num2 == 0:
            return "Error: Cannot divide by zero."
        return f"Result: {num1 / num2}"
    else:
        return f"Error: Unsupported operation '{operation}'. Use add, subtract, multiply, or divide."

@tool
def get_word_length(word: str) -> int:
    """Calculate and return the exact character count of a word or string."""
    return len(word)

# Initialize pre-built Wikipedia Community Tool
wikipedia_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500))

tools = [calculate, get_word_length, wikipedia_tool]
print(f"Initialized {len(tools)} tools: {[t.name for t in tools]}")


### Pattern 2: Agent Setup with Prompt & Scratchpad

The system prompt includes placeholders for chat_history and agent_scratchpad to store internal tool thought iterations.

In [1]:
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a versatile AI assistant equipped with calculation, word analysis, and Wikipedia search tools. Always use available tools for facts or calculations."),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, prompt)

# AgentExecutor configured to return intermediate steps and handle tool errors
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=5,
    return_intermediate_steps=True,
    handle_parsing_errors=True
)
print("Production AgentExecutor built successfully!")


### Pattern 3: Intermediate Steps Inspection & Conversational Memory

Inspecting intermediate_steps reveals the exact (AgentAction, Observation) trace generated during execution.

In [1]:
# Query with Intermediate Steps Inspection
query = "What is the length of the word 'Supercalifragilisticexpialidocious' and multiply that length by 3?"
print(f"Executing Query: {query}\n")

# Initialize chat history
chat_history = []

result = agent_executor.invoke({"input": query, "chat_history": chat_history})

print(f"\nFinal Answer: {result['output']}")
print(f"\n--- Intermediate Execution Steps ({len(result['intermediate_steps'])}) ---")
for i, (action, observation) in enumerate(result["intermediate_steps"], 1):
    print(f"Step {i}: Tool Used = '{action.tool}' | Arguments = {action.tool_input}")
    print(f"        Observation = {observation}\n")
